Export the GEO h5ad: raw and log-normalized counts, spot metadata, spatial coordinates and UMAP.
- Spot metadata: brain region, cell type probabilities (scaled per sample), raw neuron probability and neuronal-low spots.

In [ ]:
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.preprocessing import minmax_scale
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")
(base / "geo").mkdir(exist_ok=True)
adata = sc.read_h5ad(base / "adatas" / "20240104_integrated_annotated.h5ad")

In [ ]:
geo = {"733": "Foxf2iKO_R1", "734": "Foxf2iKO_R2", "738": "Foxf2iKO_R3", "740": "Foxf2iKO_R4",
       "735": "Ctrl_R1", "737": "Ctrl_R2", "739": "Ctrl_R3", "741": "Ctrl_R4"}
sex = dict.fromkeys(["733", "734", "735", "737"], "male") | dict.fromkeys(["738", "739", "740", "741"], "female")
prob = [c for c in adata.obs.columns if c.startswith("Prob ")]
mouse = adata.obs["sample"].astype(str)
obs = pd.DataFrame({
    "sample": mouse.map(geo), "mouse_id": mouse, "sex": mouse.map(sex),
    "genotype": adata.obs["condition"].astype(str).map({"KO": "Foxf2iKO", "WT": "Ctrl"}),
    "total_counts": adata.obs["total_counts"], "n_genes": adata.obs["n_genes_by_counts"],
    "brain_region": adata.obs["clusters_broad"].astype(str),
    "neuronal_low": adata.obs["Prob Neurons"] <= np.percentile(adata.obs["Prob Neurons"], 25),
    "prob_Neurons_raw": adata.obs["Prob Neurons"],
}, index=adata.obs_names)
obs[[p.replace("Prob ", "prob_").replace(" ", "_").replace("/", "_") for p in prob]] = adata.obs.groupby("sample", observed=True)[prob].transform(minmax_scale).values
obs.head()

In [ ]:
out = ad.AnnData(X=adata.layers["counts"], obs=obs, var=adata.var[["gene_ids"]], layers={"log1p_norm": adata.layers["normalized_jointly"]},
                 obsm={"spatial": adata.obsm["spatial"], "X_umap": adata.obsm["X_umap"]})
out.write_h5ad(base / "geo" / "Visium_Foxf2iKO_Ctrl.h5ad", compression="gzip")
out